# TRAIN MODEL MŨ BẢO HIỂM v2 TRÊN GOOGLE COLAB (GPU T4 MIỄN PHÍ)

### Hướng dẫn 3 bước cực nhanh:
1. Vào menu: **Thời lượng chạy (Runtime)** -> **Thay đổi loại thời lượng chạy (Change runtime type)** -> Chọn **T4 GPU** -> Bấm **Lưu (Save)**.
2. Nhìn cột bên trái, bấm vào biểu tượng **Thư mục (Files)** -> Kéo thả 2 file từ máy tính vào đây:
   - `dataset_v2_train.zip` (18.2 MB)
   - `yolo11s_helmet_lp.pt` (ở thư mục `D:\hoclamAI\giaothong\models\`)
3. Bấm **Thời lượng chạy (Runtime)** -> **Chạy tất cả (Run all)**. Đợi ~4 phút model sẽ tự động tải về máy bạn!

In [ ]:
# Kiểm tra GPU và cài đặt thư viện
!nvidia-smi
!pip install -q ultralytics

In [ ]:
import os, glob, shutil, zipfile

# Giải nén dataset vào /content/dataset_v2_train
zip_files = glob.glob('/content/**/dataset_v2_train*.zip', recursive=True)
assert len(zip_files) > 0, 'CHƯA TÌM THẤY file dataset_v2_train.zip! Vui lòng kéo thả file này vào cột Files bên trái!'

print(f'Đang giải nén: {zip_files[0]}...')
with zipfile.ZipFile(zip_files[0], 'r') as zf:
    zf.extractall('/content')

# Tự động cập nhật data.yaml
yaml_files = glob.glob('/content/**/data.yaml', recursive=True)
assert len(yaml_files) > 0, 'Không tìm thấy data.yaml trong bộ dữ liệu!'
yaml_path = yaml_files[0]
folder = os.path.dirname(os.path.abspath(yaml_path))

with open(yaml_path, 'r', encoding='utf-8') as f:
    content = f.read()

lines = []
for line in content.split('\n'):
    if line.strip().startswith('path:'):
        lines.append(f"path: '{folder}'")
    else:
        lines.append(line)

with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print('Dataset đã sẵn sàng tại:', folder)
print('data.yaml:', yaml_path)

In [ ]:
from ultralytics import YOLO
import glob

# Tìm model gốc để train tiếp
base_pts = glob.glob('/content/**/yolo11s_helmet_lp.pt', recursive=True)
base_model = base_pts[0] if base_pts else 'yolo11s.pt'
print(f'Sử dụng base model: {base_model}')

model = YOLO(base_model)

# Bắt đầu huấn luyện
results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    lr0=0.001,
    degrees=15.0,        # Góc xoay mô phỏng xe rẽ trái / rẽ phải
    perspective=0.0005,  # Nghiêng phối cảnh góc nhìn cầu vượt
    hsv_v=0.4,           # Chống chói nắng / bóng râm
    save=True,
    project='/content/runs_v2',
    name='helmet_v2'
)
print('HUẤN LUYỆN HOÀN TẤT!')

In [ ]:
import glob, shutil, os
from google.colab import files

# Tìm file model tốt nhất
pts = glob.glob('/content/runs_v2/**/best.pt', recursive=True)
if pts:
    out_pt = '/content/yolo11s_helmet_lp_v2.pt'
    shutil.copy2(pts[-1], out_pt)
    sz = os.path.getsize(out_pt) / (1024 * 1024)
    print(f'Model hoàn thiện: {out_pt} ({sz:.1f} MB)')
    print('Đang tự động tải model về máy tính của bạn...')
    files.download(out_pt)
else:
    print('Không tìm thấy best.pt!')